In [129]:
import pandas as pd
import numpy as np

# csv파일들을 불러옴
df1 = pd.read_csv('../data/tmdb_5000_credits.csv')
df2 = pd.read_csv('../data/tmdb_5000_movies.csv')
df1.columns = ['id', 'title', 'cast', 'crew']

# 두 csv파일을 하나로 합침
df = df2.merge(df1[['id', 'cast', 'crew']], on = 'id')
df.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count', 'cast', 'crew'],
      dtype='object')

In [130]:
from ast import literal_eval
# 장르의 값은 타입이 문자열
type(df['genres'][0])
# 장르의 값을 리스트로 변경
genre0 = literal_eval(df['genres'][0]) # 딕셔너리 같지만 위 코드로 확인해보면 문자열이라 literal이용해서 리스트로 변경
# genre0 # id, name 있는걸 확인

for genre in genre0:
  print(genre['name'])


Action
Adventure
Fantasy
Science Fiction


In [131]:
# cast의 값은 타입이 문자열
type(df['cast'][0])
# 배우의 값을 리스트로 변경
cast0 = literal_eval(df['cast'][0]) 
# cast0 #cast_id, character, gender, id, name, order
for cast in cast0:
  print(cast['name'])


Sam Worthington
Zoe Saldana
Sigourney Weaver
Stephen Lang
Michelle Rodriguez
Giovanni Ribisi
Joel David Moore
CCH Pounder
Wes Studi
Laz Alonso
Dileep Rao
Matt Gerald
Sean Anthony Moran
Jason Whyte
Scott Lawrence
Kelly Kilgour
James Patrick Pitt
Sean Patrick Murphy
Peter Dillon
Kevin Dorman
Kelson Henderson
David Van Horn
Jacob Tomuri
Michael Blain-Rozgay
Jon Curry
Luke Hawker
Woody Schultz
Peter Mensah
Sonia Yee
Jahnel Curfman
Ilram Choi
Kyla Warren
Lisa Roumain
Debra Wilson
Chris Mala
Taylor Kibby
Jodie Landau
Julie Lamm
Cullen B. Madden
Joseph Brady Madden
Frankie Torres
Austin Wilson
Sara Wilson
Tamica Washington-Miller
Lucy Briant
Nathan Meister
Gerry Blair
Matthew Chamberlain
Paul Yates
Wray Wilson
James Gaylyn
Melvin Leno Clark III
Carvon Futrell
Brandon Jelkes
Micah Moch
Hanniyah Muhammad
Christopher Nolen
Christa Oliver
April Marie Thomas
Bravita A. Threatt
Colin Bleasdale
Mike Bodnar
Matt Clayton
Nicole Dionne
Jamie Harrison
Allan Henry
Anthony Ingruber
Ashley Jeffery
Dean Kno

In [132]:
# keyword의 값은 타입이 문자열
type(df['keywords'][0])
# keywords의 값을 리스트로 변경
keyword0 = literal_eval(df['keywords'][0]) 
keyword0
# keyword0 # id, name
lst = []
for keyword in keyword0:
  print(keyword['name'])


culture clash
future
space war
space colony
society
space travel
futuristic
romance
space
alien
tribe
alien planet
cgi
marine
soldier
battle
love affair
anti war
power relations
mind and soul
3d


In [133]:
# x데이터가 들어오면 x데이터에서 name들만 추출하여 리스트를 만들어 반환
# 예 : cast 데이터가 들어오면 cast데이터에서 배우 이름만 리스트로 만들어 반환
def get_list(x):
	# x가 리스트인지 확인
	if isinstance(x, list):
		# names = []
		# for i in x:
		# 	names.append(i['name'])
		names = [i['name'] for i in x]
		# 처음부터 3개만 추출(최대 3개 추출)
		if len(names) > 3:
			names = names[:3] #0번지부터 3번지 전까지 
		return names
	return []

In [134]:
# genres, keywords, cast
get_list(literal_eval(df['genres'][0]))

['Action', 'Adventure', 'Fantasy']

In [135]:
features = ['genres', 'keywords', 'cast', 'crew']

# 문자열로된 값을 리스트로 변환
for feature in features:
  df[feature] = df[feature].apply(literal_eval)

In [136]:
# df['genres'][0]

In [137]:
# genres, keywords, cast를 name만 추출해서 덮어쓰기
# df['genres'] = df['genres'].apply(literal_eval)
# df[feature] = df[feature].apply(get_list)
features = ['genres', 'keywords', 'cast']
for feature in features:
	df[feature] = df[feature].apply(get_list)

In [138]:
# df['cast']

In [139]:
# cast, genres, keywords 리스트 값에 있는 공백을 제거
# 공백을 제거하지 않고 나~중에 학습을 하면
# 배우 성때문에 해당 배우와 상관없는 영화가 추천되거나
# 공상 과학에서 과학때문에 순수 과학 영화가 추천되는 상황이 발생할 수 있음


In [140]:
sample = df['cast'][0]
print(sample)
result = [str.lower(i.replace(' ', '')) for i in sample]
print(result)
result = [i.replace(' ', '') for i in sample]
print(result)

['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver']
['samworthington', 'zoesaldana', 'sigourneyweaver']
['SamWorthington', 'ZoeSaldana', 'SigourneyWeaver']


In [141]:
def clean_data(x):
	if isinstance(x, list):
		return [str.lower(i.replace(' ', '')) for i in x]
	elif isinstance(x, str):
		return str.lower(x.replace(' ', ''))
	else:
		return ''

In [142]:
for feature in features:
	df[feature] = df[feature].apply(clean_data)

In [143]:
df['cast'][0]

['samworthington', 'zoesaldana', 'sigourneyweaver']

In [144]:
jobs = [i['job'] for i in df['crew'][0]]
# jobs # Director : 감독

In [145]:
# 0번 영화 Avatar의 스탭들 중 감독을 조회
for crew in df['crew'][0]:
  if crew['job'] == 'Director':
    print(crew['name'])

James Cameron


In [146]:
def get_director(x):
  for crew in x: 
    if crew['job'] == 'Director':
      return crew['name']
  return np.nan

In [147]:
get_director(df['crew'][0])

'James Cameron'

In [148]:
df['director'] = df['crew'].apply(get_director)
df['director'] = df['director'].apply(clean_data)

In [149]:
df['director'].isnull().sum()

np.int64(0)

In [150]:
# 감독, 배우, 키워드, 장를르 하나의 열에 통합

x = df.loc[0]
res = f'{' '.join(x['keywords'])} {' '.join(x['cast'])} {' '.join(x['genres'])} {x['director']}'
res
# print(' '.join(x['keywords']))
# print(' '.join(x['cast']))
# print(' '.join(x['genres']))
# print(x['director'])

'cultureclash future spacewar samworthington zoesaldana sigourneyweaver action adventure fantasy jamescameron'

In [151]:
# 감독, 배우, 키워드, 장르를 하나의 열에 통합하는 함수
def create_soup(x):
  return f'{' '.join(x['keywords'])} {' '.join(x['cast'])} {' '.join(x['genres'])} {x['director']}'

In [152]:
df['soup'] = df.apply(create_soup, axis=1)
df['soup']

0       cultureclash future spacewar samworthington zo...
1       ocean drugabuse exoticisland johnnydepp orland...
2       spy basedonnovel secretagent danielcraig chris...
3       dccomics crimefighter terrorist christianbale ...
4       basedonnovel mars medallion taylorkitsch lynnc...
                              ...                        
4798    unitedstates–mexicobarrier legs arms carlosgal...
4799     edwardburns kerrybishé marshadietlein comedy ...
4800    date loveatfirstsight narration ericmabius kri...
4801       danielhenney elizacoupe billpaxton  danielhsia
4802    obsession camcorder crush drewbarrymore brianh...
Name: soup, Length: 4803, dtype: object

In [153]:
from sklearn.feature_extraction.text import CountVectorizer

count = CountVectorizer()
count_matrix = count.fit_transform(df['soup'])

In [154]:
# 코사인 유사도 계산
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim = cosine_similarity(count_matrix, count_matrix)

In [155]:
# 영화 제목이 주어지면 주어진 영화 제목과 내용이 유사한 영화 10개를 추천하는 함수
indices = pd.Series(df.index, index=df['title']).drop_duplicates()
def get_recommendations(title, cosine_sim=cosine_sim):
  # 영화 제목으로 idx를 찾음(왜? 해당 영화의 코사인 유사도를 가져오기 위해)
  # 코사인 유사도는 idx를 이용해서 찾아야 함
  idx = indices[title]
  # 코사인 유사도를 정렬하려고 하는데 그냥 정렬하면 기존 위치가 섞임
  # -> 유사도만 보면 어떤 영화인지 모름
  # -> 코사인 유사도를 (idx, 코사인유사도값)으로 된 튜플을 만들면 섞여도 idx를 알 수 있음
  sim_scores = list(enumerate(cosine_sim[idx])) 	# 영화 Avatart의 코사인 유사도
  # 코사인 유사도를 기준으로 정렬(람다함수 활용)
  sim_scores = sorted(sim_scores, key=lambda x : x[1], reverse=True)
  # 0번지는 검색영화이기 때문에 0번지를 제외한 10개 추출
  sim_scores = sim_scores[1:11]
  # 여기서 부터는 코사인 유사도 값이 필요 없기 때문에 영화 idx만 추출해서 리스트로 만들기
  movie_indices = [i[0] for i in sim_scores] 
  
	# 영화 idx를 이용해서 영화 제목을 가져와서 반환
  return df['title'].iloc[movie_indices]

In [156]:
print(get_recommendations('Avatar', cosine_sim))

206                         Clash of the Titans
71        The Mummy: Tomb of the Dragon Emperor
786                           The Monkey King 2
131                                     G-Force
466                            The Time Machine
715                           The Scorpion King
1      Pirates of the Caribbean: At World's End
5                                  Spider-Man 3
9            Batman v Superman: Dawn of Justice
10                             Superman Returns
Name: title, dtype: object
